In [2]:
# Import modules, custom functions, datax
import os, shutil; shutil.copy("/Users/henry.el-jawhari/github/msc_thesis/initialise.py", os.getcwd()); from initialise import *

/opt/miniconda3/envs/venv_thesis/lib/python3.11/site-packages/dalex/_global_checks.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Data is taken from DrivenData's DengAI: Predicting Disease Spread competition 
https://www.drivendata.org/competitions/44/dengai-predicting-disease-spread/

In [3]:
data_types = {
    "city": "category"
    , "week_start_date": "datetime64[ns]"
    , "year": "Int64"
    , "weekofyear": "Int64"

    , "station_max_temp_c": "float64"
    , "station_min_temp_c": "float64"
    , "station_avg_temp_c": "float64"
    , "station_precip_mm": "float64"
    , "station_diur_temp_rng_c": "float64"

    , "precipitation_amt_mm": "float64"
    
    , "reanalysis_sat_precip_amt_mm": "float64"
    , "reanalysis_dew_point_temp_k": "float64"
    , "reanalysis_air_temp_k": "float64"
    , "reanalysis_relative_humidity_percent": "float64"
    , "reanalysis_specific_humidity_g_per_kg": "float64"
    , "reanalysis_precip_amt_kg_per_m2": "float64"
    , "reanalysis_max_air_temp_k": "float64"
    , "reanalysis_min_air_temp_k": "float64"
    , "reanalysis_avg_temp_k": "float64"
    , "reanalysis_tdtr_k": "float64"

    , "ndvi_se": "float64"
    , "ndvi_sw": "float64"
    , "ndvi_ne": "float64"
    , "ndvi_nw": "float64"

    , "total_cases": "Int64"
}

data_dictionary = {
    # City and date indicators
    "city": "City abbreviations: 'sj' for San Juan and 'iq' for Iquitos"
    , "week_start_date": "Date given in yyyy-mm-dd format"
    , "year": "Year of obervation"
    , "weekofyear": "Week number for the year"

    # NOAA's GHCN daily climate data weather station measurements
    , "station_max_temp_c": "Maximum temperature"
    , "station_min_temp_c": "Minimum temperature"
    , "station_avg_temp_c": "Average temperature"
    , "station_precip_mm": "Total precipitation"
    , "station_diur_temp_rng_c": "Diurnal temperature range"

    # PERSIANN satellite precipitation measurements (0.25 x 0.25 degree scale)
    , "precipitation_amt_mm": "Total precipitation"
    
    # NOAA's NCEP Climate Forecast System Reanalysis measurements (0.5 x 0.5 degree scale)
    , "reanalysis_sat_precip_amt_mm": "Total precipitation"
    , "reanalysis_dew_point_temp_k": "Mean dew point temperature"
    , "reanalysis_air_temp_k": "Mean air temperature"
    , "reanalysis_relative_humidity_percent": "Mean relative humidity"
    , "reanalysis_specific_humidity_g_per_kg": "Mean specific humidity"
    , "reanalysis_precip_amt_kg_per_m2": "Total precipitation"
    , "reanalysis_max_air_temp_k": "Maximum air temperature"
    , "reanalysis_min_air_temp_k": "Minimum air temperature"
    , "reanalysis_avg_temp_k": "Average air temperature"
    , "reanalysis_tdtr_k": "Diurnal temperature range"

    # Satellite vegetation - Normalized difference vegetation index (NDVI) - NOAA's CDR Normalized Difference Vegetation Index (0.5 x 0.5 degree scale) measurements
    , "ndvi_se": "Pixel southeast of city centroid"
    , "ndvi_sw": "Pixel southwest of city centroid"
    , "ndvi_ne": "Pixel northeast of city centroid"
    , "ndvi_nw": "Pixel northwest of city centroid"

    , "total_cases": "Int64"
}

In [4]:
df = (
    correct_dtypes(pd.read_csv("data/dengue_features_train.csv"), data_types)
    .merge(correct_dtypes(pd.read_csv("data/dengue_labels_train.csv"), data_types))
    )

In [5]:
# Initial inspection of data
df.head()

,city,year,weekofyear,week_start_date,ndvi_ne,ndvi_nw,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,...,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_avg_temp_c,station_diur_temp_rng_c,station_max_temp_c,station_min_temp_c,station_precip_mm,total_cases
0,sj,1990,18,1990-04-30,0.122600,0.103725,0.198483,0.177617,12.42,297.572857,...,73.365714,12.42,14.012857,2.628571,25.442857,6.900000,29.4,20.0,16.0,4
1,sj,1990,19,1990-05-07,0.169900,0.142175,0.162357,0.155486,22.82,298.211429,...,77.368571,22.82,15.372857,2.371429,26.714286,6.371429,31.7,22.2,8.6,5
2,sj,1990,20,1990-05-14,0.032250,0.172967,0.157200,0.170843,34.54,298.781429,...,82.052857,34.54,16.848571,2.300000,26.714286,6.485714,32.2,22.8,41.4,4
3,sj,1990,21,1990-05-21,0.128633,0.245067,0.227557,0.235886,15.36,298.987143,...,80.337143,15.36,16.672857,2.428571,27.471429,6.771429,33.3,23.3,4.0,3
4,sj,1990,22,1990-05-28,0.196200,0.262200,0.251200,0.247340,7.52,299.518571,...,80.460000,7.52,17.210000,3.014286,28.942857,9.371429,35.0,23.9,5.8,6


In [31]:
df.shape

(1456, 25)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1456 entries, 0 to 1455
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   city                                   1456 non-null   category      
 1   year                                   1456 non-null   Int64         
 2   weekofyear                             1456 non-null   Int64         
 3   week_start_date                        1456 non-null   datetime64[ns]
 4   ndvi_ne                                1262 non-null   float64       
 5   ndvi_nw                                1404 non-null   float64       
 6   ndvi_se                                1434 non-null   float64       
 7   ndvi_sw                                1434 non-null   float64       
 8   precipitation_amt_mm                   1443 non-null   float64       
 9   reanalysis_air_temp_k                  1446 non-null   float64 

In [7]:
# For the purpose of eda we can drop `Response ID`
prettyDescribe(df)

1,456 observations | 25 features


,min,mean,max,std,25%,50%,75%,na_values
year,1990.0,2001.031593,2010.0,5.408314,1997.0,2002.0,2005.0,0 (0.00%)
weekofyear,1.0,26.503434,53.0,15.019437,13.75,26.5,39.25,0 (0.00%)
ndvi_ne,-0.40625,0.142294,0.508357,0.140531,0.04495,0.128817,0.248483,194 (13.32%)
ndvi_nw,-0.4561,0.130553,0.454429,0.119999,0.049217,0.121429,0.2166,52 (3.57%)
ndvi_se,-0.015533,0.203783,0.538314,0.07386,0.155087,0.19605,0.248846,22 (1.51%)
ndvi_sw,-0.063457,0.202305,0.546017,0.083903,0.144209,0.18945,0.246982,22 (1.51%)
precipitation_amt_mm,0.0,45.760388,390.6,43.715537,9.8,38.34,70.235,13 (0.89%)
reanalysis_air_temp_k,294.635714,298.701852,302.2,1.36242,297.658929,298.646429,299.833571,10 (0.69%)
reanalysis_avg_temp_k,294.892857,299.225578,302.928571,1.261715,298.257143,299.289286,300.207143,10 (0.69%)
reanalysis_dew_point_temp_k,289.642857,295.246356,298.45,1.52781,294.118929,295.640714,296.46,10 (0.69%)


In [32]:
# Inspect our target variable `full vaccine`
fig = (px
 .line(df, x = "week_start_date", y = "total_cases", color = "city",
       labels = {"week_start_date": "Date", "total_cases": "Total Cases", "city": "City"},
       title = "Number of Dengue Cases /week")
#  .update_layout(title = dict(text = "Full Vaccine", x = 0.5))
)

fig.show()

In [33]:
for city in set(df["city"]):
    to_plot = df[df["city"] == city].corr(numeric_only = True)
    px.imshow(to_plot,
              height = 800, width = 1000, 
              color_continuous_scale = px.colors.sequential.Viridis, title = f"Correlation for city: {city}"
              ).update_xaxes(tickangle = 45).show()

In [26]:
# Feature correlation with our response variable `full vaccine`
cases_corr = (df.groupby("city")
.corr(numeric_only = True)[["total_cases"]]
.reset_index()
.rename(columns = {"level_1": "var"})
.query("var != 'total_cases'")
# .pivot(columns = "city", index = "var", values = "total_cases")
# .reset_index()
# .assign(abs_diff = lambda x: np.abs(x["iq"] - x["sj"]))
# .sort_values("abs_diff", ascending = False)
.reset_index(drop = True)
)

cases_corr

,city,var,total_cases
0,iq,year,0.179451
1,iq,weekofyear,-0.011850
2,iq,ndvi_ne,0.020215
3,iq,ndvi_nw,-0.009586
4,iq,ndvi_se,-0.041067
5,iq,ndvi_sw,0.032999
6,iq,precipitation_amt_mm,0.090171
7,iq,reanalysis_air_temp_k,0.097098
8,iq,reanalysis_avg_temp_k,0.079872
9,iq,reanalysis_dew_point_temp_k,0.230401


In [27]:
df.columns

Index(['city', 'year', 'weekofyear', 'week_start_date', 'ndvi_ne', 'ndvi_nw',
       'ndvi_se', 'ndvi_sw', 'precipitation_amt_mm', 'reanalysis_air_temp_k',
       'reanalysis_avg_temp_k', 'reanalysis_dew_point_temp_k',
       'reanalysis_max_air_temp_k', 'reanalysis_min_air_temp_k',
       'reanalysis_precip_amt_kg_per_m2',
       'reanalysis_relative_humidity_percent', 'reanalysis_sat_precip_amt_mm',
       'reanalysis_specific_humidity_g_per_kg', 'reanalysis_tdtr_k',
       'station_avg_temp_c', 'station_diur_temp_rng_c', 'station_max_temp_c',
       'station_min_temp_c', 'station_precip_mm', 'total_cases'],
      dtype='object')

In [35]:
to_plot = [
    ['ndvi_ne', 'ndvi_nw', 'ndvi_se', 'ndvi_sw', 'precipitation_amt_mm'],
    ['reanalysis_air_temp_k', 'reanalysis_avg_temp_k', 'reanalysis_dew_point_temp_k', 'reanalysis_max_air_temp_k', 'reanalysis_min_air_temp_k',
     'reanalysis_precip_amt_kg_per_m2', 'reanalysis_relative_humidity_percent', 'reanalysis_sat_precip_amt_mm', 'reanalysis_specific_humidity_g_per_kg', 'reanalysis_tdtr_k'],
    ['station_avg_temp_c', 'station_diur_temp_rng_c', 'station_max_temp_c', 'station_min_temp_c', 'station_precip_mm']
    ]
to_exclude = ["city", "year", "weekofyear", "week_start_date", "total_cases"]

In [37]:
# for vars in to_plot:
#     px.scatter_matrix(
#         df,
#         dimensions = vars, # [x for x in df.columns if x not in to_exclude],
#         color = "city", opacity = 0.25,
#         height = 1000, width = 1000
#     ).update_traces(diagonal_visible = False).show()

In [39]:
# Inspect the 3 highest and lowest correlated features with `cases_corr`
# for var in list(cases_corr["var"][:3]) + list(cases_corr["var"][-3:]):

#     fig = px.scatter(df.astype({"total_cases": "float64", var: "float64"}), x = "total_cases", y = var, 
#                      color = "city", facet_col = "city",
#                      trendline = "ols").update_xaxes(matches = None)
#     fig.show()

In [ ]:
with open('fig_dat.pickle', 'wb') as file:
    pickle.dump(fig.data, file, protocol = pickle.HIGHEST_PROTOCOL)